In [104]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
ROOT = Path.cwd().parent.parent
sys.path.append(str(ROOT))
from src.loading_data.load_data import get_data
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [105]:
def check_num_obs(X, year):
    no_obs = []
    num_zeroes = (X[X== 0].count()/len(X)).sort_values(ascending=False) 
    for idx, value in enumerate(num_zeroes):
        if value == 1.0:
            no_obs.append(num_zeroes.index[idx])
    if len(no_obs) > 0:
        print(f'Year: {year}. Columns found to have zero observations:')
        print(no_obs)
        return no_obs
    else:
        print(f'Year: {year}. All columns have observations')
        return []

In [ ]:
def generate_coeficients() -> pd.DataFrame:
    df = get_data()
    dont_encode = ['nchild', 'active', 'MEMS7_ALL', 'year']
    discrete_variables = [c for c in df.columns if c not in dont_encode]
    reference_categories = [f'{c}_{df[c].value_counts().index[0]}' for c in discrete_variables]
    encoder = OneHotEncoder(drop='first', handle_unknown = 'ignore', sparse_output = False).set_output(transform='pandas')
    encoded = encoder.fit_transform(df[discrete_variables])
    df_encoded = pd.concat([df, encoded], axis=1).drop(columns=discrete_variables+['MEMS7_ALL']) # dropped gender= other as coef=0
    all_years = []
    for year in sorted(df_encoded['year'].unique()):
        df_encoded_sy = df_encoded[df_encoded['year'] == year]
        df_encoded_sy = df_encoded_sy.drop(columns=['year'])
        Y = df_encoded_sy['active']
        X = df_encoded_sy.drop(columns=['active'])
        scaler = StandardScaler()
        continuous_cols = ['nchild']
        X[continuous_cols] = scaler.fit_transform(X[continuous_cols])
        no_obs = check_num_obs(X, year)
        if len(no_obs) > 0:
            X =  X.drop(columns = no_obs)
        logistic = sm.Logit(Y, X)
        output = logistic.fit()
        logistic_results = pd.DataFrame({
            'feature_names' : output.params.index,
            'odds_ratios' : np.exp(output.params.values),
            'pvalues' : output.pvalues.values,
            'confidence_lower' : np.exp(output.conf_int()[0].values),
            'confidence_upper' : np.exp(output.conf_int()[1].values),
            'year' : year 
        })
        all_years.append(logistic_results)
    all = pd.concat(all_years)
    save_path = Path(ROOT / 'data' / 'logistic_coefficients')
    all.to_csv(save_path / 'logistic_coef_results.csv', index=False)
    print(f'Saved Logistic Coefficient Results to\n:>>>{save_path}')
    return 'done'

In [107]:
df = generate_coeficients()
df.head()

DataFrame Cleaned Sucessfully...
DataFrame Information:
>>> Columns:
Index(['Gend3', 'Disab3', 'Age9', 'Eth7', 'NSSEC5', 'Educ6', 'MEMS7_ALL',
       'year', 'IMD10', 'nchild', 'VolAny', 'Motiva_POP', 'motivb_POP',
       'motivc_POP', 'motivd_POP', 'WorkStat10', 'active'],
      dtype='str')
>>> Shape (86089, 17)
--------------------------
Columns NOT in Master.csv:
>>> ['nadults', 'motivex2a', 'motivex2b', 'motivex2c', 'motivex2d', 'inclus_a', 'inclus_b', 'inclus_c', 'comm2', 'limfreti1', 'limfreti2', 'limfreti3', 'limfreti4', 'limfreti5', 'limfreti6', 'limfreti7', 'limfreti8']
Unhealthy Columns:
>>> ['motive_POP', 'comm1', 'anxious', 'happy', 'lifesat', 'lone', 'worthw', 'indev', 'indevtry']
Year: 2016/17. Columns found to have zero observations:
['Gend3_3.0']
Optimization terminated successfully.
         Current function value: 0.544852
         Iterations 6
Year: 2017/18. Columns found to have zero observations:
['Age9_8.0', 'Age9_9.0', 'NSSEC5_5.0']
Optimization terminated succe

,feature_names,odds_ratios,pvalues,confidence_lower,confidence_upper,year
0,nchild,0.873943,1.298801e-09,0.836721,0.912820,2016/17
1,Gend3_2.0,0.933548,9.568823e-02,0.861008,1.012201,2016/17
2,Disab3_2.0,1.714241,2.045489e-12,1.475113,1.992135,2016/17
3,Disab3_3.0,1.539972,3.948261e-12,1.363166,1.739709,2016/17
4,Age9_3.0,1.163621,9.049569e-02,0.976366,1.386789,2016/17


In [57]:
df = get_data()
df.head()

DataFrame Cleaned Sucessfully...
DataFrame Information:
>>> Columns:
Index(['Gend3', 'Disab3', 'Age9', 'Eth7', 'NSSEC5', 'Educ6', 'MEMS7_ALL',
       'year', 'IMD10', 'nchild', 'VolAny', 'Motiva_POP', 'motivb_POP',
       'motivc_POP', 'motivd_POP', 'WorkStat10', 'active'],
      dtype='str')
>>> Shape (86089, 17)
--------------------------
Columns NOT in Master.csv:
>>> ['nadults', 'motivex2a', 'motivex2b', 'motivex2c', 'motivex2d', 'inclus_a', 'inclus_b', 'inclus_c', 'comm2', 'limfreti1', 'limfreti2', 'limfreti3', 'limfreti4', 'limfreti5', 'limfreti6', 'limfreti7', 'limfreti8']
Unhealthy Columns:
>>> ['motive_POP', 'comm1', 'anxious', 'happy', 'lifesat', 'lone', 'worthw', 'indev', 'indevtry']


,Gend3,Disab3,Age9,Eth7,NSSEC5,Educ6,MEMS7_ALL,year,IMD10,nchild,VolAny,Motiva_POP,motivb_POP,motivc_POP,motivd_POP,WorkStat10,active
19887,1.0,3.0,2.0,4.0,1.0,1.0,0.0,2016/17,2.0,1.0,1.0,1.0,2.0,2.0,4.0,2.0,False
19888,1.0,3.0,4.0,4.0,2.0,1.0,90.0,2016/17,4.0,0.0,0.0,2.0,2.0,4.0,4.0,2.0,False
19889,2.0,3.0,4.0,1.0,1.0,1.0,1200.0,2016/17,7.0,0.0,0.0,2.0,2.0,2.0,4.0,1.0,True
19890,1.0,3.0,4.0,2.0,1.0,1.0,0.0,2016/17,4.0,1.0,0.0,2.0,3.0,4.0,4.0,1.0,False
19891,2.0,1.0,4.0,2.0,1.0,1.0,30.0,2016/17,4.0,1.0,0.0,1.0,1.0,2.0,3.0,2.0,False


In [58]:
dont_encode = ['nchild', 'active', 'MEMS7_ALL', 'year']
discrete_variables = [c for c in df.columns if c not in dont_encode]
# df_encoded = pd.get_dummies(df, columns = discrete_variables, drop_first=True )
for c in discrete_variables:
    print(c, df[c].value_counts().index[0])

reference_categories = [f'{c}_{df[c].value_counts().index[0]}' for c in discrete_variables]
reference_categories

Gend3 2.0
Disab3 3.0
Age9 4.0
Eth7 1.0
NSSEC5 1.0
Educ6 1.0
IMD10 2.0
VolAny 0.0
Motiva_POP 2.0
motivb_POP 2.0
motivc_POP 2.0
motivd_POP 4.0
WorkStat10 1.0


['Gend3_2.0',
 'Disab3_3.0',
 'Age9_4.0',
 'Eth7_1.0',
 'NSSEC5_1.0',
 'Educ6_1.0',
 'IMD10_2.0',
 'VolAny_0.0',
 'Motiva_POP_2.0',
 'motivb_POP_2.0',
 'motivc_POP_2.0',
 'motivd_POP_4.0',
 'WorkStat10_1.0']

In [59]:
encoder = OneHotEncoder(drop='first', handle_unknown = 'ignore', sparse_output = False).set_output(transform='pandas')

In [60]:
encoded = encoder.fit_transform(df[discrete_variables])
df_encoded = pd.concat([df, encoded], axis=1).drop(columns=discrete_variables+['MEMS7_ALL', 'Gend3_3.0'])
df_encoded.head()

,year,nchild,active,Gend3_2.0,Disab3_2.0,Disab3_3.0,Age9_3.0,Age9_4.0,Age9_5.0,Age9_6.0,...,motivd_POP_5.0,WorkStat10_2.0,WorkStat10_3.0,WorkStat10_4.0,WorkStat10_5.0,WorkStat10_6.0,WorkStat10_7.0,WorkStat10_8.0,WorkStat10_9.0,WorkStat10_10.0
19887,2016/17,1.0,False,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19888,2016/17,0.0,False,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19889,2016/17,0.0,True,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19890,2016/17,1.0,False,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19891,2016/17,1.0,False,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [61]:
df_encoded_2016 = df_encoded[df_encoded['year'] == '2017/18']
df_encoded_2016 = df_encoded_2016.drop(columns=['year'])

In [62]:
Y = df_encoded_2016['active']
X = df_encoded_2016.drop(columns=['active'])

In [63]:
corr = X.corr()

In [64]:
high_corr = [(col1, col2) for col1 in corr.columns 
             for col2 in corr.columns 
             if col1 != col2 and abs(corr[col1][col2]) == 1.0]
print(high_corr)

[]


In [65]:
print(X.var()[X.var() < 100])

nchild             0.806878
Gend3_2.0          0.246166
Disab3_2.0         0.118161
Disab3_3.0         0.181610
Age9_3.0           0.164861
                     ...   
WorkStat10_6.0     0.042487
WorkStat10_7.0     0.017705
WorkStat10_8.0     0.049416
WorkStat10_9.0     0.004224
WorkStat10_10.0    0.027047
Length: 61, dtype: float64


In [66]:
import numpy as np
rank = np.linalg.matrix_rank(X)
print(f"Rank: {rank}, Columns: {X.shape[1]}")
# if rank < columns you have collinearity

Rank: 58, Columns: 61


In [67]:
import numpy as np

# QR decomposition reveals which column is the problem
Q, R = np.linalg.qr(X)
diag = np.abs(np.diag(R))
print(dict(zip(X.columns, diag)))

{'nchild': np.float64(115.72380913191546), 'Gend3_2.0': np.float64(76.27566992547881), 'Disab3_2.0': np.float64(39.480700653600096), 'Disab3_3.0': np.float64(66.49299359456013), 'Age9_3.0': np.float64(44.632224834362454), 'Age9_4.0': np.float64(42.30086337889073), 'Age9_5.0': np.float64(39.378806114021515), 'Age9_6.0': np.float64(35.88582605520418), 'Age9_7.0': np.float64(31.207660830911415), 'Age9_8.0': np.float64(0.0), 'Age9_9.0': np.float64(0.0), 'Eth7_2.0': np.float64(41.67656237685609), 'Eth7_3.0': np.float64(34.92236171566798), 'Eth7_4.0': np.float64(26.493131847477525), 'Eth7_5.0': np.float64(15.435424624657704), 'Eth7_6.0': np.float64(19.274268899613144), 'Eth7_7.0': np.float64(17.269111554204407), 'NSSEC5_2.0': np.float64(43.250941933700894), 'NSSEC5_3.0': np.float64(29.870038345516114), 'NSSEC5_4.0': np.float64(26.017763188579767), 'NSSEC5_5.0': np.float64(0.0), 'Educ6_2.0': np.float64(34.31647078919332), 'Educ6_3.0': np.float64(32.06208019714286), 'Educ6_4.0': np.float64(12.

In [68]:
corr

,nchild,Gend3_2.0,Disab3_2.0,Disab3_3.0,Age9_3.0,Age9_4.0,Age9_5.0,Age9_6.0,Age9_7.0,Age9_8.0,...,motivd_POP_5.0,WorkStat10_2.0,WorkStat10_3.0,WorkStat10_4.0,WorkStat10_5.0,WorkStat10_6.0,WorkStat10_7.0,WorkStat10_8.0,WorkStat10_9.0,WorkStat10_10.0
nchild,1.000000,0.019448,-0.078714,0.117938,-0.061530,0.350145,0.067455,-0.206149,-0.209236,NaN,...,0.003979,0.117606,-0.002135,0.011756,-0.214505,0.232521,-0.041041,0.004269,0.000287,-0.002357
Gend3_2.0,0.019448,1.000000,-0.019218,-0.008888,0.041943,-0.003383,-0.008953,-0.016900,-0.050844,NaN,...,0.014949,0.152782,-0.002818,0.002401,-0.036816,0.158487,0.014558,0.025764,0.024722,0.033326
Disab3_2.0,-0.078714,-0.019218,1.000000,-0.711698,-0.080869,-0.056605,-0.010893,0.070752,0.141932,NaN,...,0.003375,-0.007483,0.001357,-0.005356,0.145130,-0.030635,-0.045044,-0.034671,0.017826,0.014894
Disab3_3.0,0.117938,-0.008888,-0.711698,1.000000,0.114831,0.089521,-0.010598,-0.087317,-0.191302,NaN,...,-0.001235,0.007476,-0.012983,-0.059922,-0.182451,0.039900,-0.230637,0.050273,-0.013550,-0.035722
Age9_3.0,-0.061530,0.041943,-0.080869,0.114831,1.000000,-0.284610,-0.246960,-0.224314,-0.196240,NaN,...,0.014151,-0.095289,0.006304,-0.014105,-0.199935,0.021969,-0.028707,-0.038899,0.003630,-0.009766
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WorkStat10_6.0,0.232521,0.158487,-0.030635,0.039900,0.021969,0.067758,0.013551,-0.026455,-0.064768,NaN,...,-0.002182,-0.093734,-0.031566,-0.027497,-0.084356,1.000000,-0.029228,-0.050586,-0.014079,-0.036489
WorkStat10_7.0,-0.041041,0.014558,-0.045044,-0.230637,-0.028707,-0.033306,0.060206,0.064287,-0.039000,NaN,...,-0.001754,-0.058880,-0.019829,-0.017273,-0.052989,-0.029228,1.000000,-0.031776,-0.008844,-0.022921
WorkStat10_8.0,0.004269,0.025764,-0.034671,0.050273,-0.038899,-0.108543,-0.105436,-0.102589,-0.089750,NaN,...,0.028244,-0.101906,-0.034318,-0.029894,-0.091710,-0.050586,-0.031776,1.000000,-0.015307,-0.039670
WorkStat10_9.0,0.000287,0.024722,0.017826,-0.013550,0.003630,-0.024396,0.000672,-0.021716,-0.013699,NaN,...,0.006566,-0.028363,-0.009552,-0.008320,-0.025525,-0.014079,-0.008844,-0.015307,1.000000,-0.011041


In [69]:
X.columns

Index(['nchild', 'Gend3_2.0', 'Disab3_2.0', 'Disab3_3.0', 'Age9_3.0',
       'Age9_4.0', 'Age9_5.0', 'Age9_6.0', 'Age9_7.0', 'Age9_8.0', 'Age9_9.0',
       'Eth7_2.0', 'Eth7_3.0', 'Eth7_4.0', 'Eth7_5.0', 'Eth7_6.0', 'Eth7_7.0',
       'NSSEC5_2.0', 'NSSEC5_3.0', 'NSSEC5_4.0', 'NSSEC5_5.0', 'Educ6_2.0',
       'Educ6_3.0', 'Educ6_4.0', 'Educ6_5.0', 'Educ6_6.0', 'IMD10_2.0',
       'IMD10_3.0', 'IMD10_4.0', 'IMD10_5.0', 'IMD10_6.0', 'IMD10_7.0',
       'IMD10_8.0', 'IMD10_9.0', 'IMD10_10.0', 'VolAny_1.0', 'Motiva_POP_2.0',
       'Motiva_POP_3.0', 'Motiva_POP_4.0', 'Motiva_POP_5.0', 'motivb_POP_2.0',
       'motivb_POP_3.0', 'motivb_POP_4.0', 'motivb_POP_5.0', 'motivc_POP_2.0',
       'motivc_POP_3.0', 'motivc_POP_4.0', 'motivc_POP_5.0', 'motivd_POP_2.0',
       'motivd_POP_3.0', 'motivd_POP_4.0', 'motivd_POP_5.0', 'WorkStat10_2.0',
       'WorkStat10_3.0', 'WorkStat10_4.0', 'WorkStat10_5.0', 'WorkStat10_6.0',
       'WorkStat10_7.0', 'WorkStat10_8.0', 'WorkStat10_9.0',
       'WorkStat1

In [70]:

scaler = StandardScaler()
continuous_cols = ['nchild']
X[continuous_cols] = scaler.fit_transform(X[continuous_cols])
X[continuous_cols] = scaler.transform(X[continuous_cols])


In [71]:
model = LogisticRegression()
model.fit(X, Y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [72]:
print(len(model.coef_[0]), len(model.feature_names_in_))


61 61


In [73]:
for idx, name in enumerate(model.feature_names_in_):
    print(f'{name} : {model.coef_[0][idx]}')

nchild : -0.1380973958234691
Gend3_2.0 : -0.08094110163450981
Disab3_2.0 : 0.3826498228512277
Disab3_3.0 : 0.2739836233459544
Age9_3.0 : -0.08528180929188411
Age9_4.0 : -0.14555631326806245
Age9_5.0 : -0.10040521930523123
Age9_6.0 : -0.3023114498143779
Age9_7.0 : -0.6441333779644978
Age9_8.0 : 0.0
Age9_9.0 : 0.0
Eth7_2.0 : -0.4051136823217
Eth7_3.0 : -0.9208507806717035
Eth7_4.0 : -0.9042034707803847
Eth7_5.0 : -0.6038692888213392
Eth7_6.0 : -0.2450160431711268
Eth7_7.0 : -0.8037258708889123
NSSEC5_2.0 : -0.08796521432248418
NSSEC5_3.0 : -0.16431697952620725
NSSEC5_4.0 : -0.14541226633083304
NSSEC5_5.0 : 0.0
Educ6_2.0 : -0.16527691040220416
Educ6_3.0 : -0.2989130394360386
Educ6_4.0 : -0.6623859151471988
Educ6_5.0 : -0.32909971582024267
Educ6_6.0 : -0.6448480152866131
IMD10_2.0 : -0.09933036065292097
IMD10_3.0 : -0.03591051157930655
IMD10_4.0 : -0.07451753233478109
IMD10_5.0 : -0.1295058447455958
IMD10_6.0 : 0.01586974602106739
IMD10_7.0 : 0.11894588147825727
IMD10_8.0 : 0.0308334634050

In [81]:
y = (X == 0).mean().sort_values(ascending=False).head(10)
y.values

array([1.        , 1.        , 1.        , 0.99575787, 0.99143417,
       0.9865394 , 0.98401044, 0.98197096, 0.98001305, 0.97976831])

In [88]:
y = (X[X== 0].count()/len(X)).sort_values(ascending=False) 
y.index

Index(['Age9_9.0', 'Age9_8.0', 'NSSEC5_5.0', 'WorkStat10_9.0',
       'motivb_POP_5.0', 'Educ6_4.0', 'WorkStat10_4.0', 'WorkStat10_7.0',
       'Motiva_POP_5.0', 'Eth7_5.0', 'WorkStat10_3.0', 'Eth7_7.0',
       'WorkStat10_10.0', 'motivc_POP_5.0', 'IMD10_10.0', 'Eth7_6.0',
       'Educ6_5.0', 'Educ6_6.0', 'motivb_POP_4.0', 'WorkStat10_6.0',
       'WorkStat10_8.0', 'IMD10_8.0', 'motivd_POP_2.0', 'Eth7_4.0',
       'Motiva_POP_4.0', 'NSSEC5_4.0', 'IMD10_9.0', 'IMD10_7.0', 'NSSEC5_3.0',
       'IMD10_6.0', 'IMD10_5.0', 'Educ6_3.0', 'IMD10_4.0', 'Educ6_2.0',
       'Eth7_3.0', 'Age9_7.0', 'VolAny_1.0', 'motivb_POP_3.0',
       'WorkStat10_5.0', 'motivc_POP_4.0', 'Disab3_2.0', 'IMD10_3.0',
       'Motiva_POP_3.0', 'WorkStat10_2.0', 'Age9_6.0', 'Eth7_2.0',
       'motivd_POP_3.0', 'Age9_5.0', 'NSSEC5_2.0', 'motivc_POP_3.0',
       'IMD10_2.0', 'Age9_3.0', 'Age9_4.0', 'motivd_POP_5.0', 'motivc_POP_2.0',
       'Motiva_POP_2.0', 'motivb_POP_2.0', 'motivd_POP_4.0', 'Gend3_2.0',
       'Disab3_

In [90]:
for idx, value in enumerate(y):
    if value == 1.0:
        print(f'{y.index[idx]}, has no observations')

Age9_9.0, has no observations
Age9_8.0, has no observations
NSSEC5_5.0, has no observations


In [74]:
logit = sm.Logit(Y, X)
output = logit.fit()
print(output)

Optimization terminated successfully.
         Current function value: 0.535540
         Iterations 6


LinAlgError: Singular matrix

In [ ]:
output.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 active   No. Observations:                14656
Model:                          Logit   Df Residuals:                    14596
Method:                           MLE   Df Model:                           59
Date:                Fri, 26 Jun 2026   Pseudo R-squ.:                  0.1256
Time:                        15:07:04   Log-Likelihood:                -7982.8
converged:                       True   LL-Null:                       -9129.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
===================================================================================
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
nchild             -0.1282      0.020     -6.492      0.000      -0.167      -0.089
Gend3_2.0          -0.0712      0.041     -1.726      0.084      -0.152       0.010
Disab3_2.0          0.5318      0.077      6.893      0.000       0.381       0.683
Disab3_3.0          0.4243      0.063      6.751      0.000       0.301       0.547
Age9_3.0            0.1362      0.091      1.503      0.133      -0.041       0.314
Age9_4.0            0.1812      0.092      1.974      0.048       0.001       0.361
Age9_5.0            0.1784      0.094      1.898      0.058      -0.006       0.363
Age9_6.0            0.0472      0.100      0.473      0.637      -0.149       0.243
Age9_7.0           -0.2746      0.120     -2.284      0.022      -0.510      -0.039
Age9_8.0           -0.1602   6.27e+06  -2.56e-08      1.000   -1.23e+07    1.23e+07
Age9_9.0           -0.6295   6.27e+06     -1e-07      1.000   -1.23e+07    1.23e+07
Eth7_2.0           -0.3566      0.057     -6.284      0.000      -0.468      -0.245
Eth7_3.0           -0.9611      0.060    -15.990      0.000      -1.079      -0.843
Eth7_4.0           -0.7923      0.080     -9.885      0.000      -0.949      -0.635
Eth7_5.0           -1.0647      0.129     -8.268      0.000      -1.317      -0.812
Eth7_6.0           -0.0320      0.122     -0.262      0.793      -0.271       0.207
Eth7_7.0           -0.5548      0.117     -4.733      0.000      -0.785      -0.325
NSSEC5_2.0         -0.1650      0.054     -3.057      0.002      -0.271      -0.059
NSSEC5_3.0         -0.3511      0.084     -4.180      0.000      -0.516      -0.186
NSSEC5_4.0         -0.3876      0.148     -2.612      0.009      -0.678      -0.097
NSSEC5_5.0         -0.7898   6.27e+06  -1.26e-07      1.000   -1.23e+07    1.23e+07
Educ6_2.0          -0.1098      0.064     -1.718      0.086      -0.235       0.015
Educ6_3.0          -0.2995      0.065     -4.601      0.000      -0.427      -0.172
Educ6_4.0          -0.2847      0.164     -1.738      0.082      -0.606       0.036
Educ6_5.0          -0.3174      0.101     -3.149      0.002      -0.515      -0.120
Educ6_6.0          -0.5209      0.100     -5.230      0.000      -0.716      -0.326
IMD10_2.0           0.1676      0.070      2.403      0.016       0.031       0.304
IMD10_3.0           0.2115      0.075      2.814      0.005       0.064       0.359
IMD10_4.0           0.0324      0.079      0.412      0.680      -0.122       0.187
IMD10_5.0           0.2282      0.085      2.697      0.007       0.062       0.394
IMD10_6.0           0.3044      0.087      3.510      0.000       0.134       0.474
IMD10_7.0           0.1456      0.091      1.594      0.111      -0.033       0.325
IMD10_8.0           0.1430      0.097      1.467      0.142      -0.048       0.334
IMD10_9.0           0.0595      0.095      0.629      0.529      -0.126       0.245
IMD10_10.0          0.2785      0.131      2.118      0.034       0.021       0.536
VolAny_1.0          0.4837      0.061      7.883 

In [109]:
output.bse.values

array([1.97444948e-02, 4.12733593e-02, 7.71507731e-02, 6.28481138e-02,
       9.05726806e-02, 9.18075174e-02, 9.39946637e-02, 9.99504954e-02,
       1.20218664e-01, 6.27158256e+06, 6.27158256e+06, 5.67485960e-02,
       6.01091416e-02, 8.01590911e-02, 1.28780574e-01, 1.21775350e-01,
       1.17227034e-01, 5.39826173e-02, 8.39868137e-02, 1.48381059e-01,
       6.27158256e+06, 6.38943741e-02, 6.50992752e-02, 1.63876524e-01,
       1.00797420e-01, 9.95961110e-02, 6.97664787e-02, 7.51353453e-02,
       7.87514559e-02, 8.46099498e-02, 8.67374574e-02, 9.13714994e-02,
       9.74802437e-02, 9.45588830e-02, 1.31491759e-01, 6.13602141e-02,
       5.84147699e-02, 7.28890276e-02, 9.26266590e-02, 1.51750729e-01,
       5.79188803e-02, 7.54877397e-02, 1.06774207e-01, 2.24440381e-01,
       5.72848542e-02, 6.53246245e-02, 7.39642809e-02, 1.16872657e-01,
       1.32257049e-01, 1.16657465e-01, 1.11651319e-01, 1.10755014e-01,
       6.15455015e-02, 1.54909388e-01, 1.56127890e-01, 8.99275455e-02,
      

In [ ]:
odds_ratios = np.exp(output.params)
odds_ratios

nchild             0.879687
Gend3_2.0          0.931250
Disab3_2.0         1.702055
Disab3_3.0         1.528487
Age9_3.0           1.145862
                     ...   
WorkStat10_6.0     1.271729
WorkStat10_7.0     0.662551
WorkStat10_8.0     2.012504
WorkStat10_9.0     1.376256
WorkStat10_10.0    1.482982
Length: 61, dtype: float64

In [ ]:
feature_names = output.params.index
feature_names

Index(['nchild', 'Gend3_2.0', 'Disab3_2.0', 'Disab3_3.0', 'Age9_3.0',
       'Age9_4.0', 'Age9_5.0', 'Age9_6.0', 'Age9_7.0', 'Age9_8.0', 'Age9_9.0',
       'Eth7_2.0', 'Eth7_3.0', 'Eth7_4.0', 'Eth7_5.0', 'Eth7_6.0', 'Eth7_7.0',
       'NSSEC5_2.0', 'NSSEC5_3.0', 'NSSEC5_4.0', 'NSSEC5_5.0', 'Educ6_2.0',
       'Educ6_3.0', 'Educ6_4.0', 'Educ6_5.0', 'Educ6_6.0', 'IMD10_2.0',
       'IMD10_3.0', 'IMD10_4.0', 'IMD10_5.0', 'IMD10_6.0', 'IMD10_7.0',
       'IMD10_8.0', 'IMD10_9.0', 'IMD10_10.0', 'VolAny_1.0', 'Motiva_POP_2.0',
       'Motiva_POP_3.0', 'Motiva_POP_4.0', 'Motiva_POP_5.0', 'motivb_POP_2.0',
       'motivb_POP_3.0', 'motivb_POP_4.0', 'motivb_POP_5.0', 'motivc_POP_2.0',
       'motivc_POP_3.0', 'motivc_POP_4.0', 'motivc_POP_5.0', 'motivd_POP_2.0',
       'motivd_POP_3.0', 'motivd_POP_4.0', 'motivd_POP_5.0', 'WorkStat10_2.0',
       'WorkStat10_3.0', 'WorkStat10_4.0', 'WorkStat10_5.0', 'WorkStat10_6.0',
       'WorkStat10_7.0', 'WorkStat10_8.0', 'WorkStat10_9.0',
       'WorkStat1

In [ ]:
year = '2016/17'
logistic_results = pd.DataFrame({
    'feature_names' : output.params.index,
    'odds_ratios' : np.exp(output.params.values),
    'pvalues' : output.pvalues.values,
    'confidence_lower' : np.exp(output.conf_int()[0].values),
    'confidence_upper' : np.exp(output.conf_int()[1].values),
    'year' : year # Need to ensure that i change this year when generating df
})

C:\Users\fergu\AppData\Local\Temp\ipykernel_16060\811299504.py:7: RuntimeWarning: overflow encountered in exp
  'confidence_upper' : np.exp(output.conf_int()[1].values),


In [ ]:
logistic_results.head()

,feature_names,odds_ratios,pvalues,confidence_lower,confidence_upper,year
0,nchild,0.879687,8.446777e-11,0.846294,0.914396,2016/17
1,Gend3_2.0,0.931250,8.439068e-02,0.858883,1.009713,2016/17
2,Disab3_2.0,1.702055,5.444866e-12,1.463197,1.979905,2016/17
3,Disab3_3.0,1.528487,1.469806e-11,1.351342,1.728853,2016/17
4,Age9_3.0,1.145862,1.327645e-01,0.959481,1.368446,2016/17
